In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_DIR = Path("data/omniglot")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "images_background.zip":
        "https://github.com/brendenlake/omniglot/raw/master/python/images_background.zip",
    "images_evaluation.zip":
        "https://github.com/brendenlake/omniglot/raw/master/python/images_evaluation.zip",
}

for filename, url in files.items():
    zip_path = DATA_DIR / filename

    if not zip_path.exists():
        print(f"Downloading {filename}...")
        urlretrieve(url, zip_path)

    print(f"Extracting {filename}...")
    with ZipFile(zip_path, "r") as archive:
        archive.extractall(DATA_DIR)

print(f"Dataset saved to: {DATA_DIR.resolve()}")

Extracting images_background.zip...
Extracting images_evaluation.zip...
Dataset saved to: C:\Users\makil\Documents\ML\data\omniglot


In [ ]:
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn as nn
import torch
import torch.optim as optim

In [16]:
class OmniglotDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        image_path, alphabet, label = self.samples[index]
        img = Image.open(image_path).convert('L')
        img = self.transform(img)
        return img, alphabet, label

In [ ]:
`transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])`

root = DATA_DIR / "images_background"

alphabet_dirs = sorted(path for path in root.iterdir() if path.is_dir())
alphabet_to_idx = {
    alphabet.name: index
    for index, alphabet in enumerate(alphabet_dirs)
}

character_dirs = sorted(
    path
    for alphabet in alphabet_dirs
    for path in alphabet.iterdir()
    if path.is_dir()
)

character_to_idx = {
    character: index
    for index, character in enumerate(character_dirs)
}

samples = [
    (
        image_path,
        alphabet_to_idx[image_path.parent.parent.name],
        character_to_idx[image_path.parent],
    )
    for character in character_dirs
    for image_path in sorted(character.glob("*.png"))
]

train_dataset = OmniglotDataset(samples, transform)
train_dataset.class_to_idx = character_to_idx

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

print("Images:", len(train_dataset))
print("Classes:", len(train_dataset.class_to_idx))

images, labels_alpha, labels_char = next(iter(train_loader))

print(images.shape)        # torch.Size([32, 1, 28, 28])
print(labels_alpha.shape)  # torch.Size([32])
print(labels_char.shape)   # torch.Size([32])


Images: 19280
Classes: 964
torch.Size([32, 1, 28, 28])
torch.Size([32])
torch.Size([32])


In [27]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_layer = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.MaxPool2d(kernel_size=2),
        nn.ELU(),
        nn.Flatten(),
        nn.Linear(16*32*32, 128)
        )
        self.alphabet_layer = nn.Sequential(
        nn.Linear(30, 8),
        nn.ELU(),
        )
        self.classifier = nn.Sequential(
        nn.Linear(128 + 8, 964),
        )
        
    def forward(self, x_image, x_alphabet):
        x_image = self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet)
        x = torch.cat((x_image, x_alphabet), dim=1)
        return self.classifier(x)

In [29]:
net = Net()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for epoch in range(10):
    print(f"Epoch {epoch + 1}/10")

    for images, labels_alpha, labels_char in train_dataset:
        optimizer.zero_grad()

        outputs_alpha, outputs_char = net(images)

        loss_alpha = criterion(outputs_alpha, labels_alpha)
        loss_char = criterion(outputs_char, labels_char)

        loss = loss_alpha + loss_char
        loss.backward()
        optimizer.step()

Epoch 1/10


TypeError: Net.forward() missing 1 required positional argument: 'x_alphabet'